# Demo Pipeline — Đếm và Phân loại Phương tiện Giao thông

**Chủ đề 10** — Nhóm ……

Notebook này chạy end-to-end pipeline từ video giao thông detect + track + count thống kê + đánh giá.

Cấu trúc notebook:
1. Setup môi trường và imports
2. Khám phá dataset
3. Load model YOLOv8s đã train
4. Detect trên 1 ảnh mẫu (visualize bounding box)
5. Chạy pipeline đầy đủ trên video
6. Aggregate + vẽ biểu đồ thống kê
7. Evaluation so với baseline
8. Kết luận

> **Yêu cầu**: đã activate env `yolov8_ft`, đã có `weights/v8s_4cls_best.pt`, video test ở `data/test_videos/raw/demo_traffic.mp4`.

## 1. Setup môi trường

Chạy notebook từ thư mục root project (`~/vn_traffic_ai/`). Nếu chạy từ nơi khác, sửa `PROJECT_ROOT` bên dưới.

In [ ]:
import sys
from pathlib import Path

# Đặt project root
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Weights:", list((PROJECT_ROOT / "weights").glob("*.pt")))
print("Test videos:", list((PROJECT_ROOT / "data/test_videos/raw").glob("*.mp4")))

In [ ]:
# Kiểm tra dependency + GPU
import torch
import ultralytics
import cv2
import pandas as pd
import matplotlib.pyplot as plt

print(f"PyTorch : {torch.__version__}")
print(f"CUDA : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"OpenCV : {cv2.__version__}")

## 2. Khám phá dataset

Dataset đã merge từ 2 nguồn: **UA-DETRAC** (highway TQ) + **Vietnam Cần Thơ** (đường phố VN).
Schema thống nhất 4-class: `{0: motorcycle, 1: car, 2: bus, 3: truck}`.

In [ ]:
import yaml

with open(PROJECT_ROOT / "data/data.yaml") as f:
    data_cfg = yaml.safe_load(f)
print("Config data.yaml:")
for k, v in data_cfg.items():
    print(f" {k}: {v}")

In [ ]:
# Đếm class distribution trong training set
from collections import Counter

def count_classes(label_dir):
    c = Counter()
    for lbl in Path(label_dir).glob("*.txt"):
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    c[int(parts[0])] += 1
    return c

train_dist = count_classes(PROJECT_ROOT / "data/merged/labels/train")
val_dist = count_classes(PROJECT_ROOT / "data/merged/labels/val")

names = ["motorcycle", "car", "bus", "truck"]
print(f"{'Class':<12} {'Train':>10} {'Val':>10}")
print("-" * 34)
for i, name in enumerate(names):
    print(f"{name:<12} {train_dist[i]:>10,} {val_dist[i]:>10,}")
print(f"{'TOTAL':<12} {sum(train_dist.values()):>10,} {sum(val_dist.values()):>10,}")

In [ ]:
# Visualize distribution
fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(names))
w = 0.35
ax.bar([i - w/2 for i in x], [train_dist[i] for i in range(len(names))], w, label="Train", color="steelblue")
ax.bar([i + w/2 for i in x], [val_dist[i] for i in range(len(names))], w, label="Val", color="orange")
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_yscale("log")
ax.set_ylabel("Số instances (log scale)")
ax.set_title("Class distribution trong merged dataset")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Nhận xét**: class `car` chiếm đa số tuyệt đối (~84%), `motorcycle` chỉ 0.4% nhưng vẫn đủ để model học được nhờ chất lượng annotation VN cao (mAP motorcycle = 0.87 sau training).

## 3. Load model YOLOv8s đã train

Model được train 50 epoch trên merged dataset (5h30 phút trên GPU RTX 3500 Ada).

In [ ]:
from ultralytics import YOLO

MODEL_PATH = PROJECT_ROOT / "weights/v8s_4cls_best.pt"
BASELINE_PATH = PROJECT_ROOT / "weights/baseline_detrac4.pt"

model = YOLO(str(MODEL_PATH))
print(f"Model: {MODEL_PATH.name}")
print(f" Task : {model.task}")
print(f" Params : {sum(p.numel() for p in model.model.parameters()):,}")
print(f" Classes : {model.names}")

## 4. Detection trên 1 ảnh mẫu

Trước khi chạy trên video, thử detect trên 1 frame để kiểm tra chất lượng bounding box.

In [ ]:
import cv2
import numpy as np

# Lấy frame đầu tiên của video demo
video_path = PROJECT_ROOT / "data/test_videos/raw/demo_traffic.mp4"
cap = cv2.VideoCapture(str(video_path))
ret, frame = cap.read()
cap.release()
print(f"Frame shape: {frame.shape}")

# Predict
results = model.predict(frame, conf=0.3, verbose=False)[0]
print(f"\nSố detection: {len(results.boxes)}")
print(f"Classes có mặt: {[model.names[int(c)] for c in results.boxes.cls]}")

In [ ]:
# Vẽ bounding box lên frame
annotated = results.plot() # BGR
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title("Frame gốc")
axes[0].axis("off")

axes[1].imshow(annotated_rgb)
axes[1].set_title(f"Detection: {len(results.boxes)} đối tượng")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 5. Chạy pipeline đầy đủ trên video

Pipeline = Detection (YOLOv8s) + Tracking (ByteTrack) + Counting (line-based với direction).

Line đếm mặc định: ngang giữa frame (y = 474 với video 948px cao).

In [ ]:
from src.tracking.counter import Line
from src.pipeline.run import run_pipeline

# Định nghĩa line đếm — line ngang giữa frame demo (1764x948)
LINES = [
    Line(name="middle_horizontal",
         p1=(100, 474), p2=(1664, 474),
         count_direction="both"), # đếm cả 2 chiều
]

# Chạy pipeline (có xuất video output + CSV events)
OUT_VIDEO = PROJECT_ROOT / "results/videos/demo_notebook_out.mp4"
OUT_CSV = PROJECT_ROOT / "results/tables/demo_notebook_events.csv"
OUT_VIDEO.parent.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

events, totals, class_names = run_pipeline(
    video=str(video_path),
    weights=str(MODEL_PATH),
    lines=LINES,
    tracker="bytetrack.yaml",
    conf=0.3, iou=0.5, imgsz=512, half=True,
    out_csv=str(OUT_CSV),
    out_video=str(OUT_VIDEO),
)

print(f"\nTổng events: {len(events)}")
print(f"Totals: {totals}")
print(f"CSV lưu vào: {OUT_CSV}")
print(f"Video lưu vào: {OUT_VIDEO}")

### Xem video output có overlay counter

Chạy cell dưới nếu chạy trên máy local (Jupyter) và có display. Với JupyterLab remote, dùng HTML5 video viewer.

In [ ]:
from IPython.display import Video
Video(str(OUT_VIDEO), embed=True, width=800)

## 6. Aggregate + biểu đồ thống kê

Đọc CSV events DataFrame group theo class, thời gian, direction vẽ biểu đồ.

In [ ]:
from src.stats.aggregator import (
    events_to_df, summarize, counts_per_bucket,
    flow_rate, peak_period, cumulative_counts, counts_by_direction,
)

df = pd.read_csv(OUT_CSV)
df_enriched = events_to_df(df.to_dict("records"))
print("Head:")
print(df_enriched.head(10))
print(f"\nShape: {df_enriched.shape}")

In [ ]:
# Tổng hợp
summ = summarize(df_enriched)
print("Summary:")
for k, v in summ.items():
    print(f" {k}: {v}")

In [ ]:
# Flow rate
fr_min = flow_rate(df_enriched, unit="minute")
fr_hour = flow_rate(df_enriched, unit="hour")
print(f"Flow rate: {fr_min['total']} xe/phút hoặc {fr_hour['total']} xe/giờ")
print(f"Per class (xe/giờ): {fr_hour['per_class']}")

In [ ]:
# Biểu đồ 1: tổng theo class
per_class = summ["per_class"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(per_class.keys(), per_class.values(), color="steelblue")
for i, (k, v) in enumerate(per_class.items()):
    ax.text(i, v, str(v), ha="center", va="bottom", fontsize=11)
ax.set_ylabel("Số lượt qua line")
ax.set_title("Số lượng phương tiện theo loại")
plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ 2: cumulative theo thời gian
cum = cumulative_counts(df_enriched)
if not cum.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    cum.plot(ax=ax, linewidth=2)
    ax.set_xlabel("Thời gian (giây)")
    ax.set_ylabel("Số lượt tích lũy")
    ax.set_title("Đếm tích lũy theo thời gian")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Loại xe")
    plt.tight_layout()
    plt.show()
else:
    print("Video quá ngắn để vẽ cumulative.")

In [ ]:
# Biểu đồ 3: breakdown theo direction
dir_df = counts_by_direction(df_enriched)
if not dir_df.empty:
    print("Bảng line × direction:")
    display(dir_df)
    dir_df.plot(kind="bar", figsize=(8, 4), color=["#2ca02c", "#d62728"])
    plt.title("Lưu lượng theo chiều đi")
    plt.ylabel("Số lượt")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 7. Evaluation — so sánh với baseline

Đánh giá 2 tầng:
- **Tầng A**: mAP trên test set 56k ảnh (chuẩn ML)
- **Tầng B**: counting accuracy trên video có GT thủ công (chuẩn task)

In [ ]:
# Tầng B: counting accuracy vs GT
import json
from src.evaluation.metrics import report as eval_report

# Load GT
with open(PROJECT_ROOT / "data/test_videos/ground_truth/demo_traffic.json") as f:
    gt = json.load(f)
gt_counts = {k: v for k, v in gt["counts_by_class"].items() if v > 0}
print(f"Ground truth: {gt_counts}")

# Pred từ pipeline vừa chạy
pred_counts = df_enriched["class_name"].value_counts().to_dict()
print(f"Prediction: {pred_counts}")

rep = eval_report(pred_counts, gt_counts)
print(f"\n=== Metrics ===")
print(f"Overall Accuracy : {rep['overall_accuracy']:.3f}")
print(f"MAE : {rep['MAE']:.2f}")
print(f"MAPE : {rep['MAPE_percent']:.1f}%")
print(f"\nPer-class:")
for cls, r in rep["per_class"].items():
    print(f" {cls:<12} pred={r['pred']:3d} gt={r['gt']:3d} |err|={r['abs_err']:3d} acc={r['accuracy']:.3f}")

In [ ]:
# Tầng B cho baseline (v8n cũ)
baseline = YOLO(str(BASELINE_PATH))
baseline_events, baseline_totals, baseline_names = run_pipeline(
    video=str(video_path),
    weights=str(BASELINE_PATH),
    lines=LINES,
    tracker="bytetrack.yaml",
    conf=0.3, iou=0.5, imgsz=512, half=True,
    out_csv=None, out_video=None,
)
baseline_pred = pd.DataFrame(baseline_events)["class_name"].value_counts().to_dict()
baseline_rep = eval_report(baseline_pred, gt_counts)

print("=== So sánh ===")
print(f"{'Metric':<25} {'Baseline':>12} {'Model mới':>12}")
print("-" * 51)
print(f"{'Overall Accuracy':<25} {baseline_rep['overall_accuracy']:>12.3f} {rep['overall_accuracy']:>12.3f}")
print(f"{'MAE':<25} {baseline_rep['MAE']:>12.2f} {rep['MAE']:>12.2f}")
print(f"{'MAPE (%)':<25} {baseline_rep['MAPE_percent']:>12.1f} {rep['MAPE_percent']:>12.1f}")
print(f"\nBaseline pred: {baseline_pred}")
print(f"Model mới pred: {pred_counts}")
print(f"\n=> Baseline KHÔNG detect được xe máy (motorcycle không có trong training data).")

### Tầng A: mAP trên test set (tùy chọn — chạy hơi lâu, ~1-2 phút)

Uncomment để chạy. Yêu cầu test set 56k ảnh đã có label sẵn ở `data/merged/labels/test/`.

In [ ]:
# metrics_val = model.val(data=str(PROJECT_ROOT/'data/data.yaml'), split='test', imgsz=512, verbose=False)
# print(f"mAP@0.5 : {metrics_val.box.map50:.3f}")
# print(f"mAP@0.5-0.95 : {metrics_val.box.map:.3f}")
# print(f"Precision : {metrics_val.box.mp:.3f}")
# print(f"Recall : {metrics_val.box.mr:.3f}")

## 8. Kết luận

Notebook này đã minh hoạ toàn bộ pipeline **detect track count aggregate evaluate**:

| Thành phần | Chi tiết |
|---|---|
| Model | YOLOv8s 4-class (motorcycle/car/bus/truck) — 11M params, best.pt sau 50 epoch |
| Tracking | ByteTrack (built-in Ultralytics) |
| Counting | Line-based với direction (ltr/rtl) qua cross-product |
| Stats | Bucket thời gian linh hoạt, flow rate, peak, cumulative |
| Evaluation | 2 tầng: mAP trên 56k test ảnh + counting acc trên video GT |

**Kết quả nổi bật**:
- mAP@0.5 tổng = 0.843 trên test set 56k ảnh
- Motorcycle mAP = 0.871 — thành công nhờ merge dataset VN
- Vượt trội baseline (không có motorcycle)

**Hạn chế**:
- Video demo chỉ 5.9s, cần test thêm video dài để đánh giá đầy đủ
- Class truck yếu (0.52) do trade-off gộp van+truck

Nhóm có thể chạy lại notebook này với video khác (đổi `video_path`) và line khác (đổi `LINES`) để test.